This script categorizes the Loy et al 2023 dataset (Nucleic acid biomarkers of immune response and cell and tissue damage in children with COVID-19 and MIS-C). It also generates a tab-separated file with the CPM-normalised counts called: GSE225221_cfrna_counts_CPM.txt.


In [1]:
import pandas as pd
import numpy as np

# ==== 1. Load the file ====
df = pd.read_csv('GSE225221_cfrna_counts.tsv', sep='\t')

# The file has sample IDs in the first column and genes as columns.
# After transpose, genes will become the first column (geneID).

# ==== 2. Transpose ====
# First column becomes index so it turns into column names on transpose
df_t = df.set_index(df.columns[0]).transpose()

# After transpose:
# - Rows = genes
# - Columns = sample IDs
# Reset index so genes become a column called 'geneID'
df_t.reset_index(inplace=True)
df_t.rename(columns={'index': 'geneID'}, inplace=True)

# ==== 3. CPM normalisation (column-wise) ====
# All columns except 'geneID' are sample expression values
sample_cols = df_t.columns.drop('geneID')

# Extract numeric values (counts)
counts_only = df_t[sample_cols]

# Column sums = library sizes
col_sums = counts_only.sum(axis=0)

# Avoid division by zero
col_sums = col_sums.replace(0, np.nan)

# CPM calculation
cpm = counts_only.div(col_sums, axis=1) * 1_000_000

# Reassemble output
df_cpm = pd.concat([df_t[['geneID']], cpm], axis=1)

# ==== 4. Save output ====
df_cpm.to_csv('GSE225221_cfrna_counts_CPM.txt', sep='\t', index=False)

print("Saved CPM-normalized output as 'GSE225221_cfrna_counts_CPM.txt'.")

Saved CPM-normalized output as 'GSE225221_cfrna_counts_CPM.txt'.
